In [167]:
import datetime

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib as plt

In [ ]:
# files = [
#     "../data/reduced/2025-04-21.csv",
#     "../data/reduced/2025-04-22.csv",
#     "../data/reduced/2025-04-23.csv",
#     "../data/reduced/2025-04-27.csv",
#     "../data/reduced/2025-04-24.csv",
#     "../data/reduced/2025-04-25.csv",
#     "../data/reduced/2025-04-26.csv"
# ]

# files = [
#     "../data/sample/2025-04-21_sample.csv",
#     "../data/sample/2025-04-22_sample.csv",
#     "../data/sample/2025-04-23_sample.csv",
#     "../data/sample/2025-04-27_sample.csv",
#     "../data/sample/2025-04-24_sample.csv",
#     "../data/sample/2025-04-25_sample.csv",
#     "../data/sample/2025-04-26_sample.csv"
# ]

# dfs = []
# for file in files:
#     df = pd.read_csv(file)
#     dfs.append(df)


In [ ]:
# df = dfs[0]
df = pd.read_csv("../data/reduced/2025-04-21.csv")
df.head()

,tipo_transporte,tiene_bajada,tiempo_subida,tiempo_bajada,tiempo_etapa,comuna_subida,comuna_bajada,parada_subida,parada_bajada,dist_ruta_paraderos,dist_eucl_paraderos,x_subida,y_subida,x_bajada,y_bajada
0,BUS,1,2025-04-21 08:48:04,2025-04-21 08:50:39,155,RECOLETA,RECOLETA,T-4-19-SN-40,E-4-19-SN-55,853,825,347180,6301636,347201,6302489
1,BUS,1,2025-04-21 08:51:46,2025-04-21 08:54:58,192,RECOLETA,RECOLETA,E-4-19-SN-55,L-4-4-50-OP,1090,983,347200,6302473,346625,6303299
2,BUS,1,2025-04-21 15:34:28,2025-04-21 15:39:01,273,RECOLETA,RECOLETA,L-4-12-20-PO,E-4-295-OP-5,1127,959,346563,6303315,347168,6302522
3,METRO,0,2025-04-21 17:27:55,-,-,LAS CONDES,-,LOS DOMINICOS,-,-,-,356329,6302427,-,-
4,METRO,1,2025-04-21 09:12:16,2025-04-21 09:38:40,1584,ESTACION CENTRAL,SANTIAGO,ESTACION CENTRAL,PLAZA DE ARMAS,6240,3064,343933,6297455,346372,6299031


In [170]:
df.replace("-", np.nan, inplace=True)

In [171]:
transfo_df = df.copy()
transfo_df["tiempo_subida"] = pd.to_datetime(transfo_df["tiempo_subida"])
transfo_df["tiempo_bajada"] = pd.to_datetime(transfo_df["tiempo_bajada"])

transfo_df["tipo_transporte"] = transfo_df["tipo_transporte"].astype("category")
transfo_df["comuna_subida"] = transfo_df["comuna_subida"].astype("category")
transfo_df["comuna_bajada"] = transfo_df["comuna_bajada"].astype("category")
transfo_df["parada_bajada"] = transfo_df["parada_bajada"].astype("category")
transfo_df["parada_subida"] = transfo_df["parada_subida"].astype("category")

transfo_df["tiempo_etapa"] = transfo_df["tiempo_etapa"].astype(float)
transfo_df["x_bajada"] = transfo_df["x_bajada"].astype(float)
transfo_df["y_bajada"] = transfo_df["y_bajada"].astype(float)
transfo_df["x_subida"] = transfo_df["x_bajada"].astype(float)
transfo_df["y_subida"] = transfo_df["y_bajada"].astype(float)
transfo_df["dist_eucl_paraderos"] = transfo_df["dist_eucl_paraderos"].astype(float)
transfo_df["dist_ruta_paraderos"] = transfo_df["dist_ruta_paraderos"].astype(float)

transfo_df["tiene_bajada"] = transfo_df["tiene_bajada"].astype(bool)

transfo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4855306 entries, 0 to 4855305
Data columns (total 15 columns):
 #   Column               Dtype         
---  ------               -----         
 0   tipo_transporte      category      
 1   tiene_bajada         bool          
 2   tiempo_subida        datetime64[ns]
 3   tiempo_bajada        datetime64[ns]
 4   tiempo_etapa         float64       
 5   comuna_subida        category      
 6   comuna_bajada        category      
 7   parada_subida        category      
 8   parada_bajada        category      
 9   dist_ruta_paraderos  float64       
 10  dist_eucl_paraderos  float64       
 11  x_subida             float64       
 12  y_subida             float64       
 13  x_bajada             float64       
 14  y_bajada             float64       
dtypes: bool(1), category(5), datetime64[ns](2), float64(7)
memory usage: 371.1 MB


In [172]:
def map_time_range(date):
    time = date.time()
    if (datetime.time(00, 00, 00) <= time) and (time <= datetime.time(5, 59, 59)):
        return "early"
    elif (datetime.time(6, 00, 00) <= time) and (time <= datetime.time(11, 59, 59)):
        return "morning"
    elif (datetime.time(12, 00, 00) <= time) and (time <= datetime.time(17, 59, 59)):
        return "afternoon"
    elif (datetime.time(18, 00, 00) <= time) and (time <= datetime.time(23, 59, 59)):
        return "night"

transfo_df["time_range"] = transfo_df["tiempo_subida"].map(map_time_range)

In [173]:
def map_time_type(date):
    time = date.time()
    if (datetime.time(6, 00, 00) <= time) and (time <= datetime.time(8, 59, 59)):
        return "rush_time-morning"
    elif (datetime.time(17, 00, 00) <= time) and (time <= datetime.time(19, 59, 59)):
        return "rush_time-night"
    else:
        return "normal"

transfo_df["time_type"] = transfo_df["tiempo_subida"].map(map_time_type)
transfo_df.head()

,tipo_transporte,tiene_bajada,tiempo_subida,tiempo_bajada,tiempo_etapa,comuna_subida,comuna_bajada,parada_subida,parada_bajada,dist_ruta_paraderos,dist_eucl_paraderos,x_subida,y_subida,x_bajada,y_bajada,time_range,time_type
0,BUS,True,2025-04-21 08:48:04,2025-04-21 08:50:39,155.0,RECOLETA,RECOLETA,T-4-19-SN-40,E-4-19-SN-55,853.0,825.0,347201.0,6302489.0,347201.0,6302489.0,morning,rush_time-morning
1,BUS,True,2025-04-21 08:51:46,2025-04-21 08:54:58,192.0,RECOLETA,RECOLETA,E-4-19-SN-55,L-4-4-50-OP,1090.0,983.0,346625.0,6303299.0,346625.0,6303299.0,morning,rush_time-morning
2,BUS,True,2025-04-21 15:34:28,2025-04-21 15:39:01,273.0,RECOLETA,RECOLETA,L-4-12-20-PO,E-4-295-OP-5,1127.0,959.0,347168.0,6302522.0,347168.0,6302522.0,afternoon,normal
3,METRO,False,2025-04-21 17:27:55,NaT,NaN,LAS CONDES,NaN,LOS DOMINICOS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,afternoon,rush_time-night
4,METRO,True,2025-04-21 09:12:16,2025-04-21 09:38:40,1584.0,ESTACION CENTRAL,SANTIAGO,ESTACION CENTRAL,PLAZA DE ARMAS,6240.0,3064.0,346372.0,6299031.0,346372.0,6299031.0,morning,normal


# ¿Cómo varía el tiempo de viaje respecto la hora(hora normal y hora de punta)?

hora de punta: 6-9am 5-8pm

In [174]:
df_has_bajada = transfo_df[transfo_df["tiene_bajada"] == True]
df_has_bajada.head()

,tipo_transporte,tiene_bajada,tiempo_subida,tiempo_bajada,tiempo_etapa,comuna_subida,comuna_bajada,parada_subida,parada_bajada,dist_ruta_paraderos,dist_eucl_paraderos,x_subida,y_subida,x_bajada,y_bajada,time_range,time_type
0,BUS,True,2025-04-21 08:48:04,2025-04-21 08:50:39,155.0,RECOLETA,RECOLETA,T-4-19-SN-40,E-4-19-SN-55,853.0,825.0,347201.0,6302489.0,347201.0,6302489.0,morning,rush_time-morning
1,BUS,True,2025-04-21 08:51:46,2025-04-21 08:54:58,192.0,RECOLETA,RECOLETA,E-4-19-SN-55,L-4-4-50-OP,1090.0,983.0,346625.0,6303299.0,346625.0,6303299.0,morning,rush_time-morning
2,BUS,True,2025-04-21 15:34:28,2025-04-21 15:39:01,273.0,RECOLETA,RECOLETA,L-4-12-20-PO,E-4-295-OP-5,1127.0,959.0,347168.0,6302522.0,347168.0,6302522.0,afternoon,normal
4,METRO,True,2025-04-21 09:12:16,2025-04-21 09:38:40,1584.0,ESTACION CENTRAL,SANTIAGO,ESTACION CENTRAL,PLAZA DE ARMAS,6240.0,3064.0,346372.0,6299031.0,346372.0,6299031.0,morning,normal
5,METRO,True,2025-04-21 10:49:35,2025-04-21 11:17:20,1665.0,SANTIAGO,LA FLORIDA,PLAZA DE ARMAS,BELLAVISTA DE LA FLORIDA,11630.0,10307.0,351430.0,6289929.0,351430.0,6289929.0,morning,normal


In [175]:
df_has_bajada.groupby("time_type").agg({"tiempo_etapa": "mean"})

,tiempo_etapa
time_type,
normal,1170.222742
rush_time-morning,1336.311544
rush_time-night,1348.333420


In [176]:
df_has_bajada.groupby(["time_type", "tipo_transporte"], observed=True).agg({"tiempo_etapa": "mean"})

tiempo_etapa
time_type         tipo_transporte              
normal            BUS                910.083856
                  METRO             1404.931747
                  METROTREN          818.163365
                  ZP                1073.179436
rush_time-morning BUS               1028.553282
                  METRO             1594.244718
                  METROTREN          873.048247
                  ZP                1214.623057
rush_time-night   BUS               1135.530931
                  METRO             1516.261965
                  METROTREN          838.640985
                  ZP                1223.993743

# ¿Cómo varían la comuna subida y comuna bajada respecto la hora?
(cómo circulan las gentes.)

hora: 
early [0-6)
morning [6-12)
afternoon [12-18)
night [18-24)


In [177]:
pv_subida = pd.pivot_table(data=df_has_bajada, index='comuna_subida', columns="time_range", aggfunc="size", observed=False)
pv_bajada = pd.pivot_table(data=df_has_bajada, index='comuna_bajada', columns="time_range", aggfunc="size", observed=False)

In [178]:
pv_subida.sort_values(by="early", ascending=False).head()[["early"]]

time_range,early
comuna_subida,
PUENTE ALTO,6214
MAIPÚ,4028
LA FLORIDA,3009
SANTIAGO,2353
PUDAHUEL,1834


In [179]:
pv_bajada.sort_values(by="early", ascending=False).head()[["early"]]

time_range,early
comuna_bajada,
SANTIAGO,6020
LAS CONDES,3695
PUENTE ALTO,3132
PROVIDENCIA,2988
MAIPÚ,2595


In [180]:
pv_subida.sort_values(by="morning", ascending=False).head()[["morning"]]

time_range,morning
comuna_subida,
SANTIAGO,175416
PUENTE ALTO,113067
LAS CONDES,98578
LA FLORIDA,87651
PROVIDENCIA,87630


In [181]:
pv_bajada.sort_values(by="morning", ascending=False).head()[["morning"]]

time_range,morning
comuna_bajada,
SANTIAGO,299191
PROVIDENCIA,203518
LAS CONDES,176730
LA FLORIDA,62723
PUENTE ALTO,60844


In [182]:
pv_subida.sort_values(by="afternoon", ascending=False).head()[["afternoon"]]

time_range,afternoon
comuna_subida,
SANTIAGO,244601
PROVIDENCIA,146469
LAS CONDES,114610
LA FLORIDA,53516
PUENTE ALTO,50124


In [183]:
pv_bajada.sort_values(by="afternoon", ascending=False).head()[["afternoon"]]

time_range,afternoon
comuna_bajada,
SANTIAGO,188324
PROVIDENCIA,111758
LAS CONDES,96655
PUENTE ALTO,68111
LA FLORIDA,65686


In [184]:
pv_subida.sort_values(by="night", ascending=False).head()[["night"]]

time_range,night
comuna_subida,
SANTIAGO,150650
PROVIDENCIA,112090
LAS CONDES,93822
LA FLORIDA,34492
PUENTE ALTO,30387


In [185]:
pv_bajada.sort_values(by="night", ascending=False).head()[["night"]]

time_range,night
comuna_bajada,
SANTIAGO,95400
PUENTE ALTO,55732
LAS CONDES,50887
LA FLORIDA,46191
PROVIDENCIA,45331


# Graficar

heatmap con mapa de santiago